### Data Analysis and Visualization 

## 1. Setup and Data Load

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

df = pd.read_csv('../Data/Preprocessed/cleaned_data.csv', parse_dates=['Date'], low_memory=False)
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Quarter'] = df['Date'].dt.quarter
df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)

print(f"Shape: {df.shape}")
df.head()

Shape: (844338, 22)


,Store,Date,DayOfWeek,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,CompetitionDistanceMissing,CompetitionOpenMissing,StateHoliday,SchoolHoliday,Promo,Promo2,Promo2SinceYear,Promo2SinceWeek,PromoInterval,StoreType,Assortment,Sales,Year,Month,Quarter,WeekOfYear
0,1,2015-07-31,5,1270.0,9.0,2008.0,False,False,0,1,True,False,0.0,0.0,no_promo,c,a,8.568456,2015,7,3,31
1,2,2015-07-31,5,570.0,11.0,2007.0,False,False,0,1,True,True,2010.0,13.0,"Jan,Apr,Jul,Oct",a,a,8.710125,2015,7,3,31
2,3,2015-07-31,5,14130.0,12.0,2006.0,False,False,0,1,True,True,2011.0,14.0,"Jan,Apr,Jul,Oct",a,a,9.025696,2015,7,3,31
3,4,2015-07-31,5,620.0,9.0,2009.0,False,False,0,1,True,False,0.0,0.0,no_promo,c,c,9.546455,2015,7,3,31
4,5,2015-07-31,5,29910.0,4.0,2015.0,False,False,0,1,True,False,0.0,0.0,no_promo,a,a,8.480944,2015,7,3,31


**Note:** The dataset loads cleanly with no missing values and no duplicate rows. 

## 2. Descriptive Statistics of the Target Variable

In [2]:
print("Log-scale Sales statistics:")
print(df['Sales'].describe())

# Reverse the log-transform for a business-interpretable view
raw_sales = np.expm1(df['Sales'])
print("\nImplied original-scale Sales (expm1) statistics:")
print(raw_sales.describe().round(2))

Log-scale Sales statistics:
count    844338.000000
mean          8.757564
std           0.425278
min           3.828641
25%           8.488588
50%           8.759198
75%           9.031214
max          10.634677
Name: Sales, dtype: float64

Implied original-scale Sales (expm1) statistics:
count    844338.00
mean       6954.96
std        3103.82
min          45.00
25%        4858.00
50%        6368.00
75%        8359.00
max       41550.00
Name: Sales, dtype: float64


**Findings:**
- On the log scale, `Sales`  ranges from 3.83 to 10.63, with an average of 8.76 and not much spread around it — a tight, fairly even distribution. This confirms the log-transform was a good choice for modeling.

- Converting back to normal numbers, the average daily sales per store lands in the mid-thousands — which matches what a typical single store makes in a day, so the transform checks out.

- There are no zero or negative values left, which confirms that closed-store days (zero sales) were already removed during cleaning.

## 3. Outlier Detection (IQR Method)

In [3]:
Q1, Q3 = df['Sales'].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

outliers = df[(df['Sales'] < lower) | (df['Sales'] > upper)]
print(f"IQR bounds: [{lower:.2f}, {upper:.2f}]")
print(f"Outlier rows: {len(outliers):,} ({len(outliers) / len(df) * 100:.2f}% of data)")
print("\nOutliers broken down by StoreType:")
print(outliers['StoreType'].value_counts())

IQR bounds: [7.67, 9.85]
Outlier rows: 13,866 (1.64% of data)

Outliers broken down by StoreType:
StoreType
a    10198
d     1300
c     1263
b     1105
Name: count, dtype: int64


**Findings:** Only 1.64% of rows are outliers. That's a small, normal amount for daily sales data — most of these are just busy days like holidays or big promotions, not data errors. Since they're rare and easy to explain, we're keeping them in the dataset. They actually help the model learn how promotions and holidays affect sales.

## 4. Residual Data Quality Observations

In [4]:
print("StateHoliday raw category values and counts:")
print(df['StateHoliday'].value_counts())
print()
print("CompetitionDistanceMissing flag distribution:")
print(df['CompetitionDistanceMissing'].value_counts())
print()
print("CompetitionOpenMissing flag distribution:")
print(df['CompetitionOpenMissing'].value_counts())

StateHoliday raw category values and counts:


StateHoliday
0    843428
a       694
b       145
c        71
Name: count, dtype: int64



CompetitionDistanceMissing flag distribution:
CompetitionDistanceMissing
False    842152
True       2186
Name: count, dtype: int64

CompetitionOpenMissing flag distribution:
CompetitionOpenMissing
False    575738
True     268600
Name: count, dtype: int64


**Findings:**
- `StateHoliday` uses consistent text values  `'0'`, `'a'`, `'b'`, `'c'` — no mixed data types left over from Milestone 1.

- `CompetitionDistanceMissing` is `True` for just 2,186 rows (0.26%) — a tiny gap. Instead of guessing a value, it's flagged with a marker column, which is the right approach so the model knows "this one's unknown" rather than treating a guess as fact.

- `CompetitionOpenMissing` is `True` for 268,600 rows (31.8%) — a much bigger chunk. This isn't a data error; it just means the competitor's opening date genuinely isn't known for many stores. Flagging it (instead of making up a date) lets the model treat "unknown timing" as useful information on its own.

## 5. Correlation Analysis

In [5]:
corr_df = df[['Sales', 'DayOfWeek', 'CompetitionDistance', 'SchoolHoliday']].copy()
corr_df['Promo'] = df['Promo'].astype(int)
corr_df['Promo2'] = df['Promo2'].astype(int)

correlations = corr_df.corr()['Sales'].drop('Sales').sort_values(key=abs, ascending=False)
print("Correlation of each factor with Sales (log scale):")
print(correlations)

Correlation of each factor with Sales (log scale):
Promo                  0.404685
DayOfWeek             -0.195408
Promo2                -0.116678
SchoolHoliday          0.044185
CompetitionDistance   -0.030573
Name: Sales, dtype: float64


**Findings (ranked by strength):**
- **`Promo` (+0.40) — the biggest driver of sales by far. Running a promotion clearly and consistently boosts sales. This is the most useful lever the business has.

- **`DayOfWeek` (-0.20) — a moderate effect. Sales tend to drop later in the week, since many stores close or get less traffic on Sundays.

- **`Promo2` (-0.12) — this flag shows a weak negative link with sales, which looks strange at first. That's because `Promo2`just means "this store can run recurring promos" — not "a promo is running today." So on its own, this number is misleading. It only makes sense when checked together with `PromoInterval`.

- **`SchoolHoliday` (+0.04)** and **`CompetitionDistance` (-0.03) — both barely affect sales on their own. Competition distance in particular is a fixed trait of each store, not something that changes daily.



## 6. Hypothesis Testing — Does Promo Significantly Affect Sales?

In [6]:
promo_sales = df.loc[df['Promo'] == True, 'Sales']
no_promo_sales = df.loc[df['Promo'] == False, 'Sales']

t_stat, p_value = stats.ttest_ind(promo_sales, no_promo_sales, equal_var=False)

print(f"Mean Sales with Promo:    {promo_sales.mean():.4f}  (n={len(promo_sales):,})")
print(f"Mean Sales without Promo: {no_promo_sales.mean():.4f}  (n={len(no_promo_sales):,})")
print(f"\nWelch's t-test:  t = {t_stat:.2f},  p-value = {p_value:.2e}")

Mean Sales with Promo:    8.9492  (n=376,875)
Mean Sales without Promo: 8.6030  (n=467,463)

Welch's t-test:  t = 412.29,  p-value = 0.00e+00


**Findings:** Sales are clearly higher on promo days (8.95 vs. 8.60), and the gap is too big to be random chance. Promotions genuinely drive sales up — this should be a key feature in the model.

## 7. ANOVA — Does StoreType Significantly Affect Sales?

In [7]:
groups = [df.loc[df['StoreType'] == st, 'Sales'] for st in df['StoreType'].unique()]
f_stat, p_value = stats.f_oneway(*groups)

print("Mean Sales by StoreType:")
print(df.groupby('StoreType')['Sales'].agg(['mean', 'std', 'count']))
print(f"\nOne-way ANOVA:  F = {f_stat:.2f},  p-value = {p_value:.2e}")

Mean Sales by StoreType:
               mean       std   count
StoreType                            
a          8.741248  0.454458  457042
b          9.108491  0.509438   15560
c          8.763352  0.403463  112968
d          8.762754  0.361622  258768

One-way ANOVA:  F = 3827.01,  p-value = 0.00e+00


**Findings:** 

Store type `b` clearly outperforms the others — 9.11 average sales vs. ~8.75 for types `a`, `c`, and `d`, and this difference is real, not chance (p ≈ 0). Type `b` stores are rare (1.8% of the data) but consistently strong performers, worth flagging as their own feature and worth a closer look by the business (bigger stores? better locations?).

## 8. Holiday Effects on Sales

In [8]:
print("Mean Sales by StateHoliday category:")
print(df.groupby('StateHoliday')['Sales'].agg(['mean', 'count']))

print("\nMean Sales by SchoolHoliday flag:")
print(df.groupby('SchoolHoliday')['Sales'].agg(['mean', 'count']))

Mean Sales by StateHoliday category:
                  mean   count
StateHoliday                  
0             8.757459  843428
a             8.833035     694
b             8.878339     145
c             9.019219      71

Mean Sales by SchoolHoliday flag:
                   mean   count
SchoolHoliday                  
0              8.748358  680893
1              8.795918  163445


**Findings:**
- `Sales` are a bit higher on public holidays (8.83–9.02) than on regular days (8.76) — but only a small number of stores stay open on holidays, so this pattern is more of a hint than a solid conclusion.

- `SchoolHoliday` give a small, steady boost to sales (8.80 vs 8.75) — likely more family shopping trips during the day.

## 9. Seasonality Analysis

In [9]:
print("Mean Sales by Month (seasonality):")
monthly = df.groupby('Month')['Sales'].mean()
print(monthly)

print("\nMean Sales by DayOfWeek:")
print(df.groupby('DayOfWeek')['Sales'].mean())

print("\nPeak month:", monthly.idxmax(), f"({monthly.max():.3f})")
print("Lowest month:", monthly.idxmin(), f"({monthly.min():.3f})")

Mean Sales by Month (seasonality):
Month
1     8.702708
2     8.706792
3     8.759037
4     8.770073
5     8.787280
6     8.764405
7     8.761801
8     8.715198
9     8.699484
10    8.713543
11    8.799396
12    8.953880
Name: Sales, dtype: float64

Mean Sales by DayOfWeek:
DayOfWeek
1    8.919622
2    8.782726
3    8.739113
4    8.747296
5    8.797168
6    8.568214
7    8.722053
Name: Sales, dtype: float64

Peak month: 12 (8.954)
Lowest month: 9 (8.699)


**Findings:**

- **December is the peak sales month (8.95)** — classic holiday shopping. Sales then drop in January (8.70), which is a normal post-holiday slowdown.

- **Monday is the best sales day (8.92)**, and sales slowly drop through the week, hitting the lowest point on Saturday (8.57) — likely because some stores have shorter hours or are closed on Saturdays.

- **Spring months (April–June)** show a smaller secondary bump in sales — probably tied to seasonal promotions rather than a strong recurring pattern.

## 10. Long-Term Trend Analysis

In [10]:
yearly = df.groupby('Year')['Sales'].mean()
print("Mean Sales by Year:")
print(yearly)

monthly_trend = df.groupby(df['Date'].dt.to_period('M'))['Sales'].mean()
print("\nMonth-over-month trend (first 6 and last 6 periods):")
print(pd.concat([monthly_trend.head(6), monthly_trend.tail(6)]))

Mean Sales by Year:
Year
2013    8.733628
2014    8.767813
2015    8.782599
Name: Sales, dtype: float64

Month-over-month trend (first 6 and last 6 periods):
Date
2013-01    8.647261
2013-02    8.673263
2013-03    8.787695
2013-04    8.703165
2013-05    8.779926
2013-06    8.692414
2015-02    8.725048
2015-03    8.778328
2015-04    8.813055
2015-05    8.821089
2015-06    8.803773
2015-07    8.778590
Freq: M, Name: Sales, dtype: float64


**Findings:** 
- Sales have grown slowly but steadily each year: 8.73 (2013) → 8.77 (2014) → 8.78 (2015, through July) — about +0.5% per year. Combined with the strong weekly and monthly patterns found earlier, this means a good forecasting model needs to capture both this slow growth trend and the seasonal ups and downs — not treat sales as flat over time.

## 11. Summary of Key Statistical Findings

| Factor | Effect on Sales | Statistical Evidence |
|---|---|---|
| **Promo** | Strong positive (+0.40 corr.) | p ≈ 0 (Welch's t-test) — most actionable lever |
| **StoreType** | Type `b` significantly outperforms | p ≈ 0 (ANOVA) |
| **DayOfWeek** | Moderate negative; weekly cycle | Monday high, Saturday low |
| **Seasonality (Month)** | December peak, January trough | Classic holiday retail pattern |
| **Long-term trend** | Slight, steady year-over-year growth | +0.5%/yr in log-sales |
| **SchoolHoliday** | Small positive uplift | +0.05 mean log-sales |
| **StateHoliday** | Directional positive, small sample | Only 910 holiday rows total |
| **CompetitionDistance** | Weak direct correlation | Better modeled as structural, not daily, factor |
| **Promo2** | Weak/negative raw correlation | Needs interaction with `PromoInterval` to interpret |

These findings feed directly into feature engineering for the forecasting model: `Promo`,
`StoreType`, day-of-week, month, and a trend/year term are the highest-value signals identified
in this analysis.